In [ ]:
import pandas as pd
import numpy as np
bd = pd.read_csv(r'C:\Users\MSI\Downloads\PFA emna\data.csv')


In [ ]:
bd

In [ ]:
bd.shape

In [ ]:
bd.info()

In [ ]:
bd.head()

In [ ]:
bd.describe()

In [ ]:
bd.isna().sum()

In [ ]:
bd.duplicated().sum()

In [ ]:
bd = bd.drop(columns=['age'])

In [ ]:
bd

In [ ]:
import pandas as pd

# Convertir la colonne en vraie date (format jour/mois/année)
bd['m-pay_sub_birth_date'] = pd.to_datetime(bd['m-pay_sub_birth_date'], format='%d/%m/%Y')

# Calculer l'âge à partir d'aujourd'hui
bd['age'] = (pd.Timestamp.today() - bd['m-pay_sub_birth_date']).dt.days // 365

In [ ]:
bd

In [ ]:
bd['m-pay_sub_status_1_last_modif_date']=pd.to_datetime(bd['m-pay_sub_status_1_last_modif_date'])
bd['last_mvt']=(pd.Timestamp.today()-bd['m-pay_sub_status_1_last_modif_date']).dt.days
bd

In [ ]:
bd=bd.drop(columns='m-pay_sub_status_1_last_modif_date')
bd=bd.drop(columns='m-pay_sub_birth_date')
bd

In [ ]:
print(bd[bd['taux_endettement'] > 1])

In [ ]:
bd

In [ ]:
def surendettement(taux):
    if taux > 1:
        return 1
    else:
        return 0



In [ ]:
bd['surendettement'] = bd['taux_endettement'].apply(surendettement)

In [ ]:
bd

In [ ]:
bd['surendettement'].unique()

In [ ]:
ordre = {'Petite': 0, 'Moyenne': 1, 'Grande': 2}
bd['taille_entreprise'] = bd['taille_entreprise'].map(ordre)

In [ ]:
bd

In [ ]:
bd = pd.get_dummies(bd, columns=['m-pay_sub_city', 'type_contrat', 'm-pay_sub_status_1'])
bd

In [ ]:
colonnes_bool = bd.select_dtypes('bool').columns
bd[colonnes_bool] = bd[colonnes_bool].astype(int)

In [ ]:
bd

In [ ]:
bd.shape

In [ ]:
bd.info()

In [ ]:
bd.columns.tolist()

In [ ]:
bd['decision'].unique()

In [ ]:
bd_model = bd.copy()


In [ ]:
bd_model.columns.tolist()


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

labels = np.array(bd_model['decision'])
features = bd_model.drop(['decision', 'Unnamed: 0', 'm-pay_sub_msisdn', 'm-pay_sub_client_bankcode'], axis=1)
feature_list = list(features.columns)
features = np.array(features)

# Split en 3 : 60% train, 20% validation, 20% test
train_features, temp_features, train_labels, temp_labels = train_test_split(
    features, labels, test_size=0.4, random_state=42
)
val_features, test_features, val_labels, test_labels = train_test_split(
    temp_features, temp_labels, test_size=0.5, random_state=42
)

# Normalisation (pour les modèles qui en ont besoin)
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)
val_features_scaled = scaler.transform(val_features)
test_features_scaled = scaler.transform(test_features)

print('Train:', train_features.shape)
print('Validation:', val_features.shape)
print('Test:', test_features.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf.fit(train_features, train_labels)

# Évaluation sur validation
y_val_pred = clf_rf.predict(val_features)
print("=== Random Forest - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

# Évaluation sur test
y_test_pred = clf_rf.predict(test_features)
print("=== Random Forest - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.svm import SVC

clf_svm = SVC(probability=True , kernel='rbf')
clf_svm.fit(train_features_scaled, train_labels)

y_val_pred = clf_svm.predict(val_features_scaled)
print("=== SVM - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_svm.predict(test_features_scaled)
print("=== SVM - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf_knn = KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(train_features_scaled, train_labels)

y_val_pred = clf_knn.predict(val_features_scaled)
print("=== KNN - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_knn.predict(test_features_scaled)
print("=== KNN - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB

clf_nb = GaussianNB()
clf_nb.fit(train_features, train_labels)

y_val_pred = clf_nb.predict(val_features)
print("=== Naive Bayes - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_nb.predict(test_features)
print("=== Naive Bayes - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.linear_model import Perceptron

clf_perceptron = Perceptron(random_state=42)
clf_perceptron.fit(train_features_scaled, train_labels)

y_val_pred = clf_perceptron.predict(val_features_scaled)
print("=== Perceptron - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_perceptron.predict(test_features_scaled)
print("=== Perceptron - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.linear_model import SGDClassifier

clf_sgd = SGDClassifier(loss='log_loss', random_state=42)
clf_sgd.fit(train_features_scaled, train_labels)

y_val_pred = clf_sgd.predict(val_features_scaled)
print("=== SGD Classifier - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_sgd.predict(test_features_scaled)
print("=== SGD Classifier - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_lr = LogisticRegression()
clf_lr.fit(train_features_scaled, train_labels)

y_val_pred = clf_lr.predict(val_features_scaled)
print("=== Logistic Regression - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf_lr.predict(test_features_scaled)
print("=== Logistic Regression - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier()
clf.fit(train_features_scaled, train_labels)

y_val_pred = clf.predict(val_features_scaled)
print("=== Decision Tree - VALIDATION ===")
print("Accuracy:", metrics.accuracy_score(val_labels, y_val_pred))
print(metrics.confusion_matrix(val_labels, y_val_pred))
print(metrics.classification_report(val_labels, y_val_pred))

y_test_pred = clf.predict(test_features_scaled)
print("=== Decision Tree - TEST ===")
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
bd_model.shape

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
# ===== Dictionnaire des modèles + leurs données associées =====
models = {
    'Random Forest': (clf_rf, val_features),
    'Decision Tree': (clf_dt, val_features),
    'SVM': (clf_svm, val_features_scaled),
    'KNN': (clf_knn, val_features_scaled),
    'Naive Bayes': (clf_nb, val_features),
    'SGD Classifier': (clf_sgd, val_features_scaled),
    'Logistic Regression': (clf_lr, val_features_scaled),
    'DecisionTreeClassifier (scaled)': (clf, val_features_scaled),
}

plt.figure(figsize=(8, 8))

for name, (model, X_val) in models.items():
    y_proba = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(val_labels, y_proba)
    auc = roc_auc_score(val_labels, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

# Cas particulier : Perceptron n'a pas de predict_proba, on utilise decision_function
fpr_p, tpr_p, _ = roc_curve(val_labels, clf_perceptron.decision_function(val_features_scaled))
auc_p = roc_auc_score(val_labels, clf_perceptron.decision_function(val_features_scaled))
plt.plot(fpr_p, tpr_p, label=f'Perceptron (AUC = {auc_p:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Aléatoire')
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs (TPR)')
plt.title('Comparaison des courbes ROC - Tous les modèles')
plt.legend(loc='lower right')
plt.show()

In [ ]:
bd_model.shape

In [ ]:
bd_validation = bd_model.sample(n=20, random_state=42)
bd_reste = bd_model.drop(bd_validation.index)

In [ ]:
labels = np.array(bd_reste['decision'])
features = bd_reste.drop(['decision', 'Unnamed: 0', 'm-pay_sub_msisdn', 'm-pay_sub_client_bankcode'], axis=1)
feature_list = list(features.columns)
features = np.array(features)

In [ ]:
train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, test_size=0.25, random_state=42
)

print('Training Features Shape:', train_features.shape)
print('Testing Features Shape:', test_features.shape)


In [ ]:
scaler = StandardScaler()

train_features_scaled = scaler.fit_transform(train_features)
test_features_scaled = scaler.transform(test_features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

clf_rf = RandomForestClassifier(n_estimators=100, random_state=42)
clf_rf.fit(train_features, train_labels)

y_test_pred = clf_rf.predict(test_features)
print("Accuracy:", metrics.accuracy_score(test_labels, y_test_pred))
print(metrics.confusion_matrix(test_labels, y_test_pred))
print(metrics.classification_report(test_labels, y_test_pred))

In [ ]:
bd_validation.to_csv('bd_validation.csv', index=False)

In [ ]:
bd_reste.shape

In [ ]:
from sklearn.svm import SVC
from sklearn import metrics

clf_svm = SVC(kernel='rbf')
clf_svm.fit(train_features_scaled, train_labels)
y_pred = clf_svm.predict(test_features_scaled)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics

clf_knn = KNeighborsClassifier(n_neighbors=5)
clf_knn.fit(train_features_scaled, train_labels)
y_pred = clf_knn.predict(test_features_scaled)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics

clf = RandomForestClassifier(n_estimators=100)
clf.fit(train_features, train_labels)
y_pred = clf.predict(test_features)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics

clf_dt = DecisionTreeClassifier(random_state=42)
clf_dt.fit(train_features, train_labels)
y_pred = clf_dt.predict(test_features)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

clf_nb = GaussianNB()
clf_nb.fit(train_features, train_labels)
y_pred = clf_nb.predict(test_features)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.linear_model import Perceptron
from sklearn import metrics

clf = Perceptron(random_state=42)
clf.fit(train_features_scaled, train_labels)
y_pred = clf.predict(test_features_scaled)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

clf = LogisticRegression()
clf.fit(train_features_scaled, train_labels)
y_pred = clf.predict(test_features_scaled)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn import metrics

clf_sgd = SGDClassifier(random_state=42)
clf_sgd.fit(train_features_scaled, train_labels)
y_pred = clf_sgd.predict(test_features_scaled)

print("Accuracy:", metrics.accuracy_score(test_labels, y_pred))
print(metrics.confusion_matrix(test_labels, y_pred))
print(metrics.classification_report(test_labels, y_pred))

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
# ===== Dictionnaire des modèles + leurs données associées =====
models = {
    'Random Forest': (clf_rf, val_features),
    'Decision Tree': (clf_dt, val_features),
    'SVM': (clf_svm, val_features_scaled),
    'KNN': (clf_knn, val_features_scaled),
    'Naive Bayes': (clf_nb, val_features),
    'SGD Classifier': (clf_sgd, val_features_scaled),
    'Logistic Regression': (clf_lr, val_features_scaled),
}

plt.figure(figsize=(8, 8))

for name, (model, X_val) in models.items():
    y_proba = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(val_labels, y_proba)
    auc = roc_auc_score(val_labels, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

# Cas particulier : Perceptron n'a pas de predict_proba, on utilise decision_function
fpr_p, tpr_p, _ = roc_curve(val_labels, clf_perceptron.decision_function(val_features_scaled))
auc_p = roc_auc_score(val_labels, clf_perceptron.decision_function(val_features_scaled))
plt.plot(fpr_p, tpr_p, label=f'Perceptron (AUC = {auc_p:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Aléatoire')
plt.xlabel('Taux de Faux Positifs (FPR)')
plt.ylabel('Taux de Vrais Positifs (TPR)')
plt.title('Comparaison des courbes ROC - Tous les modèles')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import pickle
pickle.dump(clf_svm, open("model_svm.pkl", "wb"))

In [ ]:
import pickle
pickle.dump(clf_nb, open("model_naive_bayes.pkl", "wb"))

In [ ]:
import pickle
pickle.dump(clf_svm, open("model_logistic_regression.pkl", "wb"))